In [1]:
# full_convert_and_filter_v2.py
# Step 1: Convert ALL 10,506 XML -> YOLO
# Step 2: Filter small boxes (Updated to 16px threshold)
# Step 3: Report

import xml.etree.ElementTree as ET
import shutil
from pathlib import Path
from collections import defaultdict

# == CONFIG ====================================================================
BASE         = Path("D:/RDD2022_Japan/Japan/train")
XML_DIR      = BASE / "annotations" / "xmls"
IMG_DIR      = BASE / "images"
OUT_LABELS   = BASE / "yolo_filtered" / "labels"
OUT_IMAGES   = BASE / "yolo_filtered" / "images"

CLASS_MAP = {"D00": 0, "D10": 0, "D20": 0, "D40": 1}

# Severity filter - 600x600 images
# 16px on 600px = ~0.026 normalized
# We will remove any box smaller than 16x16 pixels
MIN_W = 0.026       # = ~16px on 600px
MIN_H = 0.026
MIN_AREA = 0.0007   # = ~16x16 / 600x600
# ==============================================================================

OUT_LABELS.mkdir(parents=True, exist_ok=True)
OUT_IMAGES.mkdir(parents=True, exist_ok=True)

stats = defaultdict(int)
class_counts = defaultdict(int)

xml_files = sorted(XML_DIR.glob("*.xml"))
print(f"Processing {len(xml_files)} XML files...")

for i, xml_path in enumerate(xml_files):
    if i % 1000 == 0:
        print(f"  {i}/{len(xml_files)} processed...")

    try:
        root   = ET.parse(xml_path).getroot()
        size   = root.find("size")
        img_w  = int(size.find("width").text)
        img_h  = int(size.find("height").text)
    except Exception:
        stats["parse_error"] += 1
        continue

    if img_w == 0 or img_h == 0:
        stats["zero_size"] += 1
        continue

    kept_lines = []

    for obj in root.findall("object"):
        cls_name = obj.find("name").text.strip()
        if cls_name not in CLASS_MAP:
            stats["unknown_class"] += 1
            continue

        bndbox = obj.find("bndbox")
        try:
            xmin = float(bndbox.find("xmin").text)
            ymin = float(bndbox.find("ymin").text)
            xmax = float(bndbox.find("xmax").text)
            ymax = float(bndbox.find("ymax").text)
        except Exception:
            stats["bad_coords"] += 1
            continue

        # Clamp to image boundaries
        xmin = max(0, min(xmin, img_w))
        xmax = max(0, min(xmax, img_w))
        ymin = max(0, min(ymin, img_h))
        ymax = max(0, min(ymax, img_h))

        # Convert to YOLO normalized format
        cx = ((xmin + xmax) / 2) / img_w
        cy = ((ymin + ymax) / 2) / img_h
        w  = (xmax - xmin) / img_w
        h  = (ymax - ymin) / img_h
        area = w * h

        stats["total_boxes"] += 1

        # Severity filter (Targeting objects < 16px)
        if w < MIN_W or h < MIN_H or area < MIN_AREA:
            stats["removed_small"] += 1
            continue

        cls_id = CLASS_MAP[cls_name]
        kept_lines.append(f"{cls_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
        class_counts[cls_id] += 1
        stats["kept_boxes"] += 1

    # Write results to text files
    if kept_lines:
        (OUT_LABELS / f"{xml_path.stem}.txt").write_text(
            "\n".join(kept_lines) + "\n"
        )
        # Copy corresponding image
        for ext in [".jpg", ".jpeg", ".png"]:
            src = IMG_DIR / (xml_path.stem + ext)
            if src.exists():
                shutil.copy2(src, OUT_IMAGES / src.name)
                stats["kept_images"] += 1
                break
    else:
        stats["removed_images"] += 1

# == Final Report ==============================================================
print("\n" + "=" * 55)
print("CONVERSION + FILTER COMPLETE (16px Threshold)")
print(f"  Total XML files    : {len(xml_files):,}")
print(f"  Kept images        : {stats['kept_images']:,}")
print(f"  Removed (no boxes) : {stats['removed_images']:,}")
print(f"  Total boxes        : {stats['total_boxes']:,}")
print(f"  Kept boxes         : {stats['kept_boxes']:,}")
print(f"  Removed (too small): {stats['removed_small']:,}")
print(f"  Cracks  (class 0)  : {class_counts[0]:,}")
print(f"  Potholes(class 1)  : {class_counts[1]:,}")
ratio = class_counts[0] / max(class_counts[1], 1)
print(f"  Crack:Pothole ratio: {ratio:.2f}:1")
print("=" * 55)

if stats['kept_images'] < 3000:
    print("WARNING: Less than 3000 images kept - consider lowering thresholds.")
elif stats['kept_images'] < 5000:
    print("ACCEPTABLE: Ready to proceed.")
else:
    print("EXCELLENT: Dataset is large enough for training.")

print(f"\nOutput folder: {OUT_LABELS.parent}")
print("NEXT STEP: Run data_stats.py to verify the new pothole count.")

Processing 10506 XML files...
  0/10506 processed...
  1000/10506 processed...
  2000/10506 processed...
  3000/10506 processed...
  4000/10506 processed...
  5000/10506 processed...
  6000/10506 processed...
  7000/10506 processed...
  8000/10506 processed...
  9000/10506 processed...
  10000/10506 processed...

CONVERSION + FILTER COMPLETE (16px Threshold)
  Total XML files    : 10,506
  Kept images        : 7,841
  Removed (no boxes) : 2,665
  Total boxes        : 16,470
  Kept boxes         : 15,784
  Removed (too small): 686
  Cracks  (class 0)  : 13,679
  Potholes(class 1)  : 2,105
  Crack:Pothole ratio: 6.50:1
EXCELLENT: Dataset is large enough for training.

Output folder: D:\RDD2022_Japan\Japan\train\yolo_filtered
NEXT STEP: Run data_stats.py to verify the new pothole count.
